# DataNexus — Exploratory Data Analysis
*FY 2024 E-Commerce Dataset · 47,231 Transactions*

In [ ]:
import sys; sys.path.insert(0,'..') 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from src.pipeline.data_generator import generate_and_save
from src.pipeline.etl import load_orders, compute_kpis, monthly_revenue, revenue_by_category

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 130

# Generate data if missing
import pathlib
if not pathlib.Path('../data/orders.csv').exists():
    generate_and_save()
df = load_orders('../data/orders.csv')
print(df.shape, df.dtypes)

## 1. Dataset Overview

In [ ]:
print(df.describe().T.to_string())
print('\nMissing values:')
print(df.isnull().sum())

## 2. Revenue Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['order_value'], bins=80, color='#378ADD', alpha=0.7, edgecolor='white')
axes[0].set_title('Order Value Distribution')
axes[0].set_xlabel('Order Value ($)')
axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

df.groupby('category')['order_value'].sum().sort_values().plot(kind='barh', ax=axes[1], color='#1D9E75')
axes[1].set_title('Revenue by Category')
axes[1].xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

plt.tight_layout()
plt.savefig('../results/figures/01_revenue_distribution.png', bbox_inches='tight')
plt.show()

## 3. Monthly Revenue Trend

In [ ]:
monthly = monthly_revenue(df)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(monthly['month_str'], monthly['gross_k'], marker='o', color='#378ADD', label='Gross Revenue', linewidth=2)
ax.plot(monthly['month_str'], monthly['net_k'],   marker='s', color='#1D9E75', label='Net Revenue',   linewidth=2, linestyle='--')
ax.fill_between(monthly['month_str'], monthly['gross_k'], alpha=0.07, color='#378ADD')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:.0f}K'))
ax.set_title('FY 2024 Monthly Revenue')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/02_monthly_revenue.png', bbox_inches='tight')
plt.show()

## 4. Order Value by Category (Box Plot)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
cat_order = df.groupby('category')['order_value'].median().sort_values(ascending=False).index
sns.boxplot(data=df[df['order_value'] < 800], x='order_value', y='category',
            order=cat_order, palette='Blues_d', ax=ax)
ax.set_title('Order Value Distribution by Category (excl. anomalies)')
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
plt.tight_layout()
plt.savefig('../results/figures/03_category_boxplot.png', bbox_inches='tight')
plt.show()

## 5. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
corr = df[['unit_price','quantity','order_value','is_anomaly']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, square=True)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../results/figures/04_correlation_heatmap.png', bbox_inches='tight')
plt.show()